<a href="https://colab.research.google.com/github/Tibodin/Resume/blob/main/%E0%B8%A3%E0%B8%B0%E0%B8%9A%E0%B8%9A_AutoML_%E0%B8%94%E0%B9%89%E0%B8%A7%E0%B8%A2_Genetic_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer, fetch_california_housing #Dataset
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore') # ซ่อน Warning เพื่อให้ผลลัพธ์ดูสะอาดตา

# ==========================================================
# 1. กำหนด Search Space สำหรับ AutoML
# ==========================================================
SEARCH_SPACE = {
    'scaler': ['None', 'Standard', 'MinMax'],
    'selector': ['None', 'PCA', 'SelectKBest'],
    'model': ['Linear', 'SVM', 'RF', 'XGB', 'NN'],

    # Hyperparameters สำหรับ Logistic/Linear Regression
    'lr_C': [0.01, 0.1, 1.0, 10.0], # สำหรับ Classification

    # Hyperparameters สำหรับ SVM
    'svm_C': [0.1, 1.0, 10.0, 100.0],
    'svm_kernel': ['linear', 'rbf'],

    # Hyperparameters สำหรับ Random Forest
    'rf_n_estimators': [10, 50, 100, 200],
    'rf_max_depth': [None, 5, 10, 20],

    # Hyperparameters สำหรับ XGBoost
    'xgb_n_estimators': [50, 100, 200],
    'xgb_learning_rate': [0.01, 0.1, 0.2],
    'xgb_max_depth': [3, 6, 9],

    # Hyperparameters สำหรับ Neural Network (MLP)
    'nn_hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'nn_learning_rate_init': [0.001, 0.01, 0.1],
    'nn_activation': ['relu', 'tanh']
}

# ==========================================================
# 2. ฟังก์ชันแปลผล Chromosome เป็น Sklearn Pipeline
# ==========================================================
def create_pipeline(chromosome, task_type='classification'):
    steps = []

    # 2.1 Preprocessing: Scaling
    if chromosome['scaler'] == 'Standard':
        steps.append(('scaler', StandardScaler()))
    elif chromosome['scaler'] == 'MinMax':
        steps.append(('scaler', MinMaxScaler()))

    # 2.2 Preprocessing: Feature Selection/Extraction
    if chromosome['selector'] == 'PCA':
        steps.append(('selector', PCA(n_components=0.9))) # เก็บ variance 90%
    elif chromosome['selector'] == 'SelectKBest':
        score_func = f_classif if task_type == 'classification' else f_regression
        steps.append(('selector', SelectKBest(score_func=score_func, k='all')))

    # 2.3 Model Selection & Hyperparameters
    model_choice = chromosome['model']
    if task_type == 'classification':
        if model_choice == 'Linear':
            model = LogisticRegression(C=chromosome['lr_C'], max_iter=1000)
        elif model_choice == 'SVM':
            model = SVC(C=chromosome['svm_C'], kernel=chromosome['svm_kernel'])
        elif model_choice == 'RF':
            model = RandomForestClassifier(n_estimators=chromosome['rf_n_estimators'], max_depth=chromosome['rf_max_depth'])
        elif model_choice == 'XGB':
            model = xgb.XGBClassifier(n_estimators=chromosome['xgb_n_estimators'], learning_rate=chromosome['xgb_learning_rate'], max_depth=chromosome['xgb_max_depth'], eval_metric='logloss')
        elif model_choice == 'NN':
            model = MLPClassifier(hidden_layer_sizes=chromosome['nn_hidden_layer_sizes'], learning_rate_init=chromosome['nn_learning_rate_init'], activation=chromosome['nn_activation'], max_iter=500)
    else: # Regression
        if model_choice == 'Linear':
            model = LinearRegression() # LR regression ไม่มี C
        elif model_choice == 'SVM':
            model = SVR(C=chromosome['svm_C'], kernel=chromosome['svm_kernel'])
        elif model_choice == 'RF':
            model = RandomForestRegressor(n_estimators=chromosome['rf_n_estimators'], max_depth=chromosome['rf_max_depth'])
        elif model_choice == 'XGB':
            model = xgb.XGBRegressor(n_estimators=chromosome['xgb_n_estimators'], learning_rate=chromosome['xgb_learning_rate'], max_depth=chromosome['xgb_max_depth'])
        elif model_choice == 'NN':
            model = MLPRegressor(hidden_layer_sizes=chromosome['nn_hidden_layer_sizes'], learning_rate_init=chromosome['nn_learning_rate_init'], activation=chromosome['nn_activation'], max_iter=500)

    steps.append(('model', model))
    return Pipeline(steps)

# ==========================================================
# 3. Objective Function (Fitness Evaluation)
# ==========================================================
def evaluate_fitness(chromosome, X, y, task_type):
    try:
        pipeline = create_pipeline(chromosome, task_type)
        if task_type == 'classification':
            # ใช้ Accuracy หรือ F1-score (ยิ่งมากยิ่งดี)
            scores = cross_val_score(pipeline, X, y, cv=3, scoring='accuracy')
            fitness = scores.mean()
        else:
            # ใช้ Negative Mean Squared Error (ค่าน้อยคือติดลบมาก เราต้องการ Maximize จึงใช้ r2 หรือ neg_mse)
            # ในที่นี้ใช้ R2 Score (ยิ่งใกล้ 1 ยิ่งดี)
            scores = cross_val_score(pipeline, X, y, cv=3, scoring='r2')
            fitness = scores.mean()

        # Constraint: หากพบโมเดลที่ทำงานแย่มาก ให้คะแนนติดลบ
        if np.isnan(fitness):
            return -999.0
        return fitness
    except Exception as e:
        # หาก Pipeline พัง (พารามิเตอร์ขัดแย้งกัน) จะให้ Fitness ต่ำสุด
        return -999.0

# ==========================================================
# 4. Metaheuristic Design: Genetic Algorithm (GA)
# ==========================================================
def generate_individual():
    """สร้างโครโมโซมแบบสุ่ม 1 ตัว"""
    return {key: random.choice(values) for key, values in SEARCH_SPACE.items()}

def crossover(parent1, parent2):
    """Uniform Crossover: สลับยีนระหว่างพ่อแม่"""
    child = {}
    for key in SEARCH_SPACE.keys():
        child[key] = parent1[key] if random.random() > 0.5 else parent2[key]
    return child

def mutate(individual, mutation_rate=0.2):
    """Mutation: สุ่มเปลี่ยนค่ายีนบางตัว"""
    mutated = individual.copy()
    for key in SEARCH_SPACE.keys():
        if random.random() < mutation_rate:
            mutated[key] = random.choice(SEARCH_SPACE[key])
    return mutated

def run_automl_ga(X, y, task_type='classification', pop_size=10, generations=5):
    print(f"--- เริ่มต้นค้นหา AutoML สำหรับ {task_type.upper()} ---")
    # 1. Initialize Population
    population = [generate_individual() for _ in range(pop_size)]
    best_overall = None
    best_fitness = -float('inf')

    for gen in range(generations):
        # 2. Evaluation
        fitness_scores = [evaluate_fitness(ind, X, y, task_type) for ind in population]

        # หา Best ของ Generation นี้
        gen_best_idx = np.argmax(fitness_scores)
        if fitness_scores[gen_best_idx] > best_fitness:
            best_fitness = fitness_scores[gen_best_idx]
            best_overall = population[gen_best_idx]

        print(f"Generation {gen+1}/{generations} | Best Fitness: {best_fitness:.4f} | Model: {best_overall['model']}")

        # 3. Selection (Tournament Selection)
        selected_population = []
        for _ in range(pop_size):
            participants = random.sample(list(zip(population, fitness_scores)), 3)
            winner = max(participants, key=lambda x: x[1])[0]
            selected_population.append(winner)

        # 4. Crossover & Mutation เพื่อสร้างรุ่นต่อไป
        next_generation = []
        for i in range(0, pop_size, 2):
            p1 = selected_population[i]
            p2 = selected_population[(i+1) % pop_size]

            c1, c2 = crossover(p1, p2), crossover(p2, p1)
            next_generation.extend([mutate(c1), mutate(c2)])

        population = next_generation

    print("\n--- ผลลัพธ์ที่ดีที่สุด ---")
    print(f"Fitness Score: {best_fitness:.4f}")
    print("Pipeline Configuration:")
    for k, v in best_overall.items():
        # แสดงเฉพาะ Hyperparameter ของโมเดลที่ถูกเลือก
        if k.split('_')[0] in ['scaler', 'selector', 'model'] or k.split('_')[0] == best_overall['model'].lower():
             print(f"  - {k}: {v}")

    return best_overall

# ==========================================================
# 5. ทดสอบรันระบบ (Experimental Results)
# ==========================================================
if __name__ == "__main__":
    # ทดสอบ 1: Classification (Breast Cancer Dataset)
    print("\n[Test 1] Classification Problem")
    data_cls = load_breast_cancer()
    X_cls, y_cls = data_cls.data, data_cls.target
    # ตั้งค่าประชากรน้อยๆ เพื่อให้รันดูเป็นตัวอย่างไวๆ (แนะนำให้เพิ่ม pop_size และ generations เวลาส่งงานจริง)
    best_cls = run_automl_ga(X_cls, y_cls, task_type='classification', pop_size=8, generations=5)

    # ทดสอบ 2: Regression (California Housing Dataset) - ใช้ข้อมูลส่วนหนึ่งเพื่อความรวดเร็ว
    print("\n[Test 2] Regression Problem")
    data_reg = fetch_california_housing()
    X_reg, y_reg = data_reg.data[:1000], data_reg.target[:1000] # สุ่มใช้ 1000 แถว
    best_reg = run_automl_ga(X_reg, y_reg, task_type='regression', pop_size=8, generations=5)


[Test 1] Classification Problem
--- เริ่มต้นค้นหา AutoML สำหรับ CLASSIFICATION ---
Generation 1/5 | Best Fitness: 0.9754 | Model: SVM
Generation 2/5 | Best Fitness: 0.9754 | Model: NN
Generation 3/5 | Best Fitness: 0.9771 | Model: NN
Generation 4/5 | Best Fitness: 0.9772 | Model: NN
Generation 5/5 | Best Fitness: 0.9772 | Model: NN

--- ผลลัพธ์ที่ดีที่สุด ---
Fitness Score: 0.9772
Pipeline Configuration:
  - scaler: Standard
  - selector: PCA
  - model: NN
  - nn_hidden_layer_sizes: (50,)
  - nn_learning_rate_init: 0.001
  - nn_activation: relu

[Test 2] Regression Problem
--- เริ่มต้นค้นหา AutoML สำหรับ REGRESSION ---
Generation 1/5 | Best Fitness: 0.4735 | Model: XGB
Generation 2/5 | Best Fitness: 0.4797 | Model: NN
Generation 3/5 | Best Fitness: 0.5620 | Model: NN
Generation 4/5 | Best Fitness: 0.5873 | Model: XGB
Generation 5/5 | Best Fitness: 0.5873 | Model: XGB

--- ผลลัพธ์ที่ดีที่สุด ---
Fitness Score: 0.5873
Pipeline Configuration:
  - scaler: Standard
  - selector: None
  - m